# Fine-tuning Médical — TechCorp Hackathon
**Modèle :** microsoft/Phi-3-mini-4k-instruct  
**Dataset :** ruslanmv/ai-medical-chatbot  
**Technique :** QLoRA (4-bit quantization + LoRA)  
**Équipe :** Inacio RODRIGUES, Théo SALY, Olivier DICK, Hayat MEGHLAT

> ⚠️ Ce modèle est **expérimental** — pas destiné à la production médicale.

**Activer le GPU :** Runtime → Changer le type d'exécution → GPU T4

In [ ]:
# Installation des dépendances
!pip install -q transformers peft accelerate bitsandbytes datasets trl

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
DATASET_NAME = "ruslanmv/ai-medical-chatbot"
OUTPUT_DIR = "./phi3-medical-lora"
MAX_SAMPLES = 2000   # Réduire pour aller plus vite (max ~250k dans le dataset)
MAX_SEQ_LEN = 512
EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-4

In [ ]:
# Chargement et préparation du dataset médical
print("Chargement du dataset médical...")
dataset = load_dataset(DATASET_NAME, split=f"train[:{MAX_SAMPLES}]")
print(f"Exemples chargés : {len(dataset)}")
print(f"Colonnes : {dataset.column_names}")
print("\nExemple :")
print(dataset[0])

In [ ]:
# Formatage au format Phi-3 chat
def format_sample(example):
    # Le dataset ruslanmv a les colonnes 'Patient' et 'Doctor'
    question = example.get("Patient", example.get("question", ""))
    answer   = example.get("Doctor",  example.get("answer", ""))
    return {
        "text": (
            "<|system|>\nYou are a helpful medical assistant. "
            "Always remind the user to consult a real doctor for diagnosis.<|end|>\n"
            f"<|user|>\n{question}<|end|>\n"
            f"<|assistant|>\n{answer}<|end|>"
        )
    }

dataset = dataset.map(format_sample, remove_columns=dataset.column_names)
print("Dataset formaté. Exemple :")
print(dataset[0]["text"][:300])

In [ ]:
# Chargement du modèle en 4-bit (QLoRA)
print(f"Chargement du modèle {MODEL_NAME}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("Modèle chargé.")

In [ ]:
# Configuration LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Entraînement
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
)

print("Démarrage de l'entraînement...")
trainer.train()
print("Entraînement terminé !")

In [ ]:
# Sauvegarde du modèle fine-tuné
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Modèle sauvegardé dans {OUTPUT_DIR}")

In [ ]:
# Test du modèle fine-tuné
from peft import PeftModel

model.eval()

test_questions = [
    "I have a headache and fever since 2 days. What should I do?",
    "What are the symptoms of diabetes?",
    "How can I lower my blood pressure naturally?",
]

for question in test_questions:
    prompt = (
        "<|system|>\nYou are a helpful medical assistant.<|end|>\n"
        f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n❓ {question}")
    print(f"💬 {response.strip()}")

## Métriques d'entraînement

Notez ici les résultats après l'entraînement :

| Métrique | Valeur |
|---|---|
| Loss initiale | |
| Loss finale | |
| Epochs | 2 |
| Samples | 2000 |
| Durée | |
| GPU | T4 |

⚠️ **Rappel** : Ce modèle est expérimental et ne remplace pas l'avis d'un médecin.